# Spike — validação do 5º pipeline (Distribution separado do ERP)

Testa os 5 simuladores na nova ordem de dependência (CRM -> ERP -> Distribution -> TMS -> Financeiro, ADR-011 adendo), com data nova (21/08/2026, sexta-feira — todos os sistemas devem operar).

Confirma: ERP gera só erp_lotes_producao; Distribution gera erp_posicoes_estoque e erp_notas_expedicao (lendo lotes do ERP e pedidos do CRM via cross-read).

In [0]:
%pip install dbldatagen Faker

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import date
from src.simuladores.simulador_factory import SimuladorFactory

data_teste = date(2026, 8, 21)

for nome_sistema in SimuladorFactory.ordem_execucao():
    simulador = SimuladorFactory.criar(nome_sistema, spark=spark, dbutils=dbutils)
    simulador.executar_seed()
    resultado = simulador.gerar_dia(data_teste)
    print(f"{resultado['sistema']}: {resultado['status']} — {resultado.get('tabelas_geradas', [])}")

In [0]:
caminho_dist = f"/Volumes/poc_pulse_observability/landing/raw/distribution/data=2026-08-21/erp_notas_expedicao.json"
df_notas = spark.read.json(caminho_dist)
df_notas.select("lote_id").show(5, truncate=False)

In [0]:
for tabela in ["erp_posicoes_estoque", "erp_notas_expedicao"]:
    spark.sql(f"DROP TABLE IF EXISTS poc_pulse_observability.bronze.{tabela}")
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_checkpoint/{tabela}", recurse=True)
    dbutils.fs.rm(f"/Volumes/poc_pulse_observability/landing/raw/_autoloader_schema/{tabela}", recurse=True)

print("Reset concluído para erp_posicoes_estoque e erp_notas_expedicao.")